# Exploratory Data Analysis

| Column | Datatype | Description |
| --- | --- | --- |
| date | string | yyyy-mm-dd |
| n_sick | int | amount of sick drivers |
| calls | float | emergency calls |
| n_duty | int | amount of **on-duty** drivers |
| n_sby | int | amount of available **standby** drivers |
| sby_need | float | amount of activated **standby** drivers |
| dafted | float | drafted **off-duty drivers** if standby drivers are not enough |

In [ ]:
# imports 
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

%matplotlib ipympl

# import dataset
df = pd.read_csv("sickness_table.csv")

# delete indexing variable of dataset
if "Unnamed: 0" in df.columns:
    del df["Unnamed: 0"]

## Evaluation of Data Quality

In [ ]:
# check if there are any empty cells
if not df.isnull().any().any():
    print("There are no empty cells in the dataset.")

# check if all floats are actually whole numbers (x.0) and parse to int
float_cols = ["calls", "sby_need", "dafted"]
parse_floats_to_ints = True
for col in float_cols:
    if not all(df[col].apply(float.is_integer)):
        print(f"Column {col} contains non-integer floats.")
        parse_floats_to_ints = False
if parse_floats_to_ints:
    print("All float columns contain only whole numbers. Parsing to int.")
    df[float_cols] = df[float_cols].astype(int)

# check if all dates follow yyyy-mm-dd schema
if all(df["date"].str.match(r"\d{4}-\d{2}-\d{2}")):
    print("All dates follow the yyyy-mm-dd schema.")
# check if data for all days is available
start_date = df["date"].min()
end_date = df["date"].max()
if not any(pd.date_range(start=start_date, end=end_date).difference(pd.to_datetime(df["date"]))):
    print("Data is available for all days.")

In [ ]:
# create profile report
profile = ProfileReport(df, title="Sickness Table Report")
profile.to_notebook_iframe()

## Feature generation

In [ ]:
# create additional date features
df["year"] = pd.DatetimeIndex(df["date"]).year
df["month"] = pd.DatetimeIndex(df["date"]).month
df["day"] = pd.DatetimeIndex(df["date"]).day
df["day_of_week"] = pd.DatetimeIndex(df["date"]).dayofweek

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["day_sin"] = np.sin(2 * np.pi * df["day"] / 31)
df["day_cos"] = np.cos(2 * np.pi * df["day"] / 31)
df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# create additional feature: n_work (actually working drivers)
df["n_work"] = df["n_duty"] - df["n_sick"] + df["sby_need"]

# percentage of sick drivers since n_duty changes
df["perc_sick"] = df["n_sick"] / df["n_duty"]

# get trend, seasonal and residual components of calls (avoiding NaNs w/ extrapolation)
result_calls = seasonal_decompose(df["calls"], model="additive", period=365)
df["calls_trend"] = result_calls.trend
df["calls_seas"] = result_calls.seasonal
df["calls_resid"] = result_calls.resid

# get trend, seasonal and residual components of perc_sick (avoiding NaNs w/ extrapolation)
result_perc_sick = seasonal_decompose(df["perc_sick"], model="additive", period=365)
df["perc_sick_trend"] = result_perc_sick.trend
df["perc_sick_seas"] = result_perc_sick.seasonal
df["perc_sick_resid"] = result_perc_sick.resid

In [ ]:
# TODO: visualize important connections/correlation
# (easily understandable, also show quality of data)

In [ ]:
# plotting decompositions to show trends, seasonality, and residuals
result_perc_sick.plot()
result_calls.plot()
plt.show()

In [ ]:
# general plotting
plt.close("all")
cols_drop = [col for col in df.columns if any(sub in col for sub in ["sin", "cos", "trend", "seas", "resid"])]
df_plot = df.drop(columns=["n_sby", "month", "day", "date", "day_of_week"] + cols_drop)  # w/o uninformative columns
fig, axs = plt.subplots(len(df_plot.columns) + 1, 1, figsize=(10, 2 * len(df_plot.columns)), sharex=True)
for i, ax in enumerate(axs):
    if i < len(axs) - 1:
        ax.plot(df_plot.iloc[:, i])
        ax.set_title(df_plot.columns[i])
    else:
        # evaluation metric: percentage of sby_need compared to n_sby
        ax.plot(100 * df["sby_need"] / df["n_sby"])
        ax.hlines(100, xmin=df.index.min(), xmax=df.index.max(), colors="r", linestyles="dashed")
        ax.set_title("Percentage of sby_need compared to n_sby")
fig.tight_layout()

In [ ]:
# plot relationship between n_work and calls for each n_duty
# better than plot in sns pairplot since noise got reduced
n_duties = df["n_duty"].unique()
fig, ax = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
for n_duty in n_duties:
    ax.scatter(
        df["n_work"].where(df["n_duty"] == n_duty),
        df["calls"].where(df["n_duty"] == n_duty),
        marker="x",
        label=f"n_duty: {n_duty}",
    )
ax.scatter(
    df["n_work"].where(df["sby_need"] > 0),
    df["calls"].where(df["sby_need"] > 0),
    alpha=0.4,
    marker=".",
    color="red",
    label="sby_need > 0",
)
ax.legend()
ax.set_xlabel("n_work")
ax.set_ylabel("calls")
ax.grid()
fig.tight_layout()

In [ ]:
# calculate correlation between n_work and calls w/o sby_need for each n_duty
print("corr between n_work and calls (only if standby drivers were needed)")
for n_duty in n_duties:
    indices = df.index[(df["sby_need"] > 0) & (df["n_duty"] == n_duty)]
    n_work = df.loc[indices, "n_work"]
    calls = df.loc[indices, "calls"]
    r = np.corrcoef(n_work, calls)
    print(f"\tFor n_duty={n_duty} the corr is {r[1, 0]}")
print("\nremaining n_work data points are solely deviating from n_duty because of n_sick")

In [ ]:
# plot correlation matrix except for n_sby since it's always 90 and other uninformative columns
cols_drop = [col for col in df.columns if any(sub in col for sub in ["sin", "cos", "trend", "resid"])]
df.drop(columns=["n_sby", "date", "n_sick", "calls", "perc_sick"] + cols_drop).corr().style.background_gradient(
    cmap="coolwarm", axis=None, vmin=-1, vmax=1
)

In [ ]:
# plot correlations (except for uninformative columns)
cols_drop = [col for col in df.columns if any(sub in col for sub in ["sin", "cos", "trend", "resid"])]
sns.pairplot(df.drop(columns=["n_sby", "n_sick", "n_duty", "dafted", "date"] + cols_drop), height=1.5)

In [ ]:
# plot ACF and PACF of calls and perc_sick
lags = 40
fig, axs = plt.subplots(2, 2, figsize=(8, 4))
plot_acf(df["calls"], lags=lags, ax=axs[0, 0])
axs[0, 0].set_title("ACF of Calls")
plot_pacf(df["calls"], lags=lags, ax=axs[0, 1])
axs[0, 1].set_title("PACF of Calls")
plot_acf(df["perc_sick"], lags=lags, ax=axs[1, 0])
axs[1, 0].set_title("ACF of Percentage Sick")
plot_pacf(df["perc_sick"], lags=lags, ax=axs[1, 1])
axs[1, 1].set_title("PACF of Percentage Sick")
axs[0, 0].grid()
axs[0, 1].grid()
axs[1, 0].grid()
axs[1, 1].grid()
fig.tight_layout()

In [ ]:
# plot FFT of calls and perc_sick
print(
    "Calls Peaks [1/d]:"
    + "\n\t- 0.14241: weekly"
    + "\n\t- 0.099: 10 days"
    + "\n\t- 0.066: half month"
    + "\n\t- 0.033: monthly"
    + "\n\t- 0.00521: half year"
    + "\n\t- 0.0026: yearly"
)
print("Perc_sick Peaks [1/d]:" + "\n\t- 0.033: monthly" + "\n\t- 0.00524: half year" + "\n\t- 0.0026: yearly")
fig, axs = plt.subplots(1, 2, figsize=(8, 2))
axs[0].magnitude_spectrum(df["calls_seas"], Fs=1)
axs[0].set_title("FFT of calls_seas")
axs[1].magnitude_spectrum(df["perc_sick_seas"], Fs=1)
axs[1].set_title("FFT of perc_sick_seas")
axs[0].grid()
axs[1].grid()
fig.tight_layout()